# Capstone A: Model a function from data

This notebook starts from `capstone/a_function_from_data/dataset.csv` (regenerate via `make_dataset.py` if needed). Work through each section, adding markdown narrative where TODO hints appear, and keep your outputs reproducible.

## Setup

Import numpy, pandas, matplotlib, and the repository utilities. Set a figure style and mark the data path.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from calculus_with_python import differentiation, integration

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("capstone/a_function_from_data/dataset.csv")


## Load the noisy samples

Read the CSV and store `x` and `y` arrays. This provides the raw dataset for modeling.

In [ ]:

data = pd.read_csv(DATA_PATH)
x_data = data["x"].to_numpy()
y_data = data["y"].to_numpy()
data.head()


## Fit a simple polynomial model

Use a low-degree polynomial to smooth the measurements. Track the RMSE so you can report how well the model performs.

In [ ]:

degree = 4
coeffs = np.polyfit(x_data, y_data, degree)
poly = np.poly1d(coeffs)
residual = y_data - poly(x_data)
rmse = np.sqrt(np.mean(residual ** 2))
print(f"Degree {degree} RMSE: {rmse:.3f}")


## Raw samples vs fitted curve

Confirm that the polynomial captures the smooth behavior while ignoring noise. This plot satisfies the raw vs fitted curve requirement.

In [ ]:

fit_grid = np.linspace(x_data.min(), x_data.max(), 400)
fit_y = poly(fit_grid)

plt.figure()
plt.plot(x_data, y_data, "o", label="Raw samples", alpha=0.5)
plt.plot(fit_grid, fit_y, "-", label="Polynomial fit", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Raw data vs fitted curve")
plt.legend()
plt.tight_layout()
plt.savefig("assets/figures/a_function_from_data/raw_vs_fit.png")


## Derivative estimate and critical points

Evaluate the derivative of the fitted polynomial and locate zero crossings. Note the probable extremum locations in the markdown explanation (TODO).

In [ ]:

analysis_grid = np.linspace(1.0, 9.0, 200)
derivative_est = differentiation.difference_quotient(poly, analysis_grid, h=1e-2)
crit_idx = np.where(np.diff(np.sign(derivative_est)) != 0)[0]
critical_points = [analysis_grid[idx] for idx in crit_idx]
print("Critical point candidates:", critical_points)

plt.figure()
plt.plot(analysis_grid, derivative_est, label="Estimated derivative")
plt.axhline(0, color="black", linewidth=0.5)
for cp in critical_points:
    plt.axvline(cp, color="red", linestyle="--", alpha=0.6)
plt.title("Derivative estimate and critical points")
plt.xlabel("x")
plt.ylabel("Derivative")
plt.legend()
plt.tight_layout()
plt.savefig("assets/figures/a_function_from_data/derivative_and_critical_points.png")


## Error vs step size h

Plot how the derivative error behaves as you shrink `h` around a central location. Summarize the convergence characteristics where the TODO marker appears.

In [ ]:

x0 = 5.0
true_derivative = poly.deriv()(x0)
h_values = np.logspace(-4, -1, 20)
errors = [abs(differentiation.difference_quotient(poly, x0, h=h) - true_derivative) for h in h_values]

plt.figure()
plt.loglog(h_values, errors, "-o")
plt.xlabel("h")
plt.ylabel("Absolute error")
plt.title("Error in difference quotient at x=5")
plt.grid(True, which="both")
plt.tight_layout()
plt.savefig("assets/figures/a_function_from_data/error_vs_h.png")


## Area estimates over subintervals

Compute trapezoidal estimates from the noisy samples and compare them with Simpson's rule on the polynomial. Document how the smoothing affects area in the TODO block.

In [ ]:

subintervals = [(1.0, 3.5), (4.5, 8.5)]
for a, b in subintervals:
    mask = (x_data >= a) & (x_data <= b)
    area_trap = integration.trapezoid(y_data[mask], x_data[mask])
    area_poly = integration.integrate_f(poly, a, b, method="simpson")
    print(f"Area over [{a}, {b}]: trapezoid = {area_trap:.4f}, polynomial simpson = {area_poly:.4f}")

plt.figure()
plt.plot(fit_grid, fit_y, label="Polynomial fit")
for a, b in subintervals:
    plt.fill_between(fit_grid, fit_y, where=(fit_grid >= a) & (fit_grid <= b), alpha=0.25, label=f"Area {a}-{b}")
plt.scatter(x_data, y_data, s=10, alpha=0.4, label="Raw samples")
plt.legend()
plt.title("Area regions and fitted curve")
plt.tight_layout()
plt.savefig("assets/figures/a_function_from_data/area_regions.png")


## Reflection

TODO: Summarize what the fitting, derivative, and area findings tell you about the underlying function. Mention how the numerical diagnostics build confidence.